In [99]:
######## ignore the print statemet created for checking row numbers, tested and should be correct
def apply_baseline_correction2(df, eeg_columns, fixation_id=101):
    """
    Applies baseline correction with accurate file row numbering.
    Now correctly matches file rows (header=row 0, first data=row 1)
    """
    df_corrected = df.copy()
    trial_ids = df["stimID"].unique()
    epsilon = 1e-10

    # Convert DataFrame indices to 1-based file rows
    def get_file_row_numbers(indices):
        return [i+1 for i in indices]  # +1 because Python is 0-based but files count from 1

    print(f"\n=== Starting Baseline Correction ===")
    print(f"First 3 columns: {df.columns[:3].tolist()}")
    print(f"Header row = Row 0 (column names)")
    print(f"First data row = Row 1")

    # 1. Compute GLOBAL baseline means
    global_fixation_mask = df["stimID"] == fixation_id
    global_fixation_rows = df[global_fixation_mask]
    print(f"\nGlobal baseline using {len(global_fixation_rows)} fixation rows at file index: {get_file_row_numbers(global_fixation_rows.index)}")
    global_baseline_means = global_fixation_rows[eeg_columns].mean(axis=0)
    
    # 2. Process each trial
    for trial_id in trial_ids:
        trial_mask = df["stimID"] == trial_id
        trial_rows = df[trial_mask]
        
        print(f"\nProcessing stimID: {trial_id}")
        print(f"Found {len(trial_rows)} rows at file rows: {get_file_row_numbers(trial_rows.index)}")
        
        if len(trial_rows) > 0:
            first_row = trial_rows.iloc[0]
            actual_file_row = trial_rows.index[0] + 2  # +1 for header, +1 for 0-based to 1-based
            print(f"First row data: {first_row[:3].to_dict()}")
            print(f"Actual file row: {actual_file_row}")  # Now matches exact file row

        if not trial_mask.any():
            print("→ No rows found, skipping")
            continue
        
        # CASE 1: Special trials → GLOBAL baseline
        if trial_id in {99, 100}:
            print(f"→ Using GLOBAL baseline (all fixations)")
            for channel in eeg_columns:
                df_corrected.loc[trial_mask, channel] = 10 * np.log10(
                    (df.loc[trial_mask, channel] + epsilon) / 
                    (global_baseline_means[channel] + epsilon))
        
        # CASE 2: Fixation trials → Skip
        elif trial_id == fixation_id:
            print("→ Fixation trial, skipping correction")
        
        # CASE 3: Normal trials → Preceding fixation
        else:
            trial_start_idx = trial_rows.index[0]
            fixation_rows = df.loc[
                (df.index < trial_start_idx+1) & 
                (df["stimID"] == fixation_id)
            ].tail(2)
            
            fixation_row_nums = get_file_row_numbers(fixation_rows.index)
            print(f"→ Preceding fixation at file rows: {fixation_row_nums}")
            print(f"→ Trial actually starts at file row: {trial_start_idx+1}")  # DEBUG
            
            if len(fixation_rows) > 0:
                trial_baseline = fixation_rows[eeg_columns].mean(axis=0)
                print("→ Applying trial-specific baseline correction")
                for channel in eeg_columns:
                    df_corrected.loc[trial_mask, channel] = 10 * np.log10(
                        (df.loc[trial_mask, channel] + epsilon) / 
                        (trial_baseline[channel] + epsilon))
            else:
                print("→ WARNING: No fixation found! Using global baseline")
                for channel in eeg_columns:
                    df_corrected.loc[trial_mask, channel] = 10 * np.log10(
                        (df.loc[trial_mask, channel] + epsilon) / 
                        (global_baseline_means[channel] + epsilon))
    
    print("\n=== Correction Complete ===")
    return df_corrected

In [77]:
import os
import pandas as pd

# Define paths
input_dir = r"\\......\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals v2"
output_dir = r"\\......\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals v2\post baseline correction"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get all CSV files in input directory
csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

# Process each file
for csv_file in csv_files:
    # Load data
    input_path = os.path.join(input_dir, csv_file)
    df = pd.read_csv(input_path)
    
    # Identify EEG columns (e.g., columns containing "Channel" or specific prefixes)
    eeg_columns = [col for col in df.columns if "Channel" in col] 
    
    # Apply trial-specific baseline correction
    df_corrected = apply_baseline_correction2(df, eeg_columns)
    
    # Save results
    output_path = os.path.join(output_dir, f"corrected_{csv_file}")
    df_corrected.to_csv(output_path, index=False)
    print(f"Processed and saved: {output_path}")

print("All files processed successfully!")


=== Starting Baseline Correction ===
First 3 columns: ['Time (s)', 'q1_y', 'Channel 1 - Delta Power']
Header row = Row 0 (column names)
First data row = Row 1

Global baseline using 72 fixation rows at file rows: [231, 232, 247, 248, 265, 266, 282, 283, 301, 302, 318, 319, 335, 336, 350, 351, 369, 370, 390, 391, 406, 407, 428, 429, 445, 446, 460, 461, 475, 476, 491, 492, 507, 508, 524, 525, 543, 544, 561, 562, 581, 582, 596, 597, 612, 613, 628, 629, 643, 644, 659, 660, 669, 670, 687, 688, 703, 704, 721, 722, 737, 738, 756, 757, 774, 775, 793, 794, 808, 809, 823, 824]

Processing stimID: 99.0
Found 60 rows at file rows: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
First row data: {'Time (s)': 51, 'q1_y': 'long_happy', 'Channel 1 - Delta Power': 7.930397974053385}
Actual file row: 1
→ Using GLOBAL basel